In [7]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%matplotlib widget
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch
import torch.optim as optim
import torch.nn as nn
from core.benchmarks import *
from core.CardiacCTdataset import DataLoaderFactory
from core.CNNmodel import *
from core.benchmarks import *
import pandas as pd
from core.CVsplits import *
from tqdm.notebook import tqdm
import logging
from core.Log import *

OUTER_FOLDS = 5; INNER_FOLDS = 3


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# Create Holdout dataset for FINAL Model Evaluation
from sklearn.model_selection import train_test_split
from core.Log import load_dataset_info, save_dataset_info


main_dataset = load_dataset_info(file="data/data_info.json")
labels = [lbl['label'] for lbl in main_dataset]
main_training, final_test = train_test_split(main_dataset,
											 test_size=22,     # 33 for 4 OUTER folds, 22 for 5 OUTER folds
											 stratify=labels,
											 random_state=42)  # 67 FOR 5 OUTER FOLDS
print(len(final_test))

for sample in main_dataset:
	if sample in final_test: sample['pool'] = 'holdout'
	else: sample['pool'] = 'main'
save_dataset_info(main_dataset, file="NCV_5_3_folds/data_info_5-3NCV.json")
#"data/data_info_5-3NCV.json"


22
Successfully saved data to: NCV_5_3_folds/data_info_5-3NCV.json


In [8]:
from core.CVsplits import create_folds_stats

create_folds_stats(OUTER_K=5, INNER_K=3)


Loaded NCV_5_3_folds/data_info_5-3NCV.json.
Generating and saving fold indices...
OUTER FOLD 0 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 1 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 2 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 3 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER

In [ ]:

# Define the parameter grids for each benchmark model
MLP_PARAM_GRID = [
	{"paramID": 101, "LR": 3e-4,  "WD": 1e-5, "DR": 0.4},
	#{"paramID": 102, "LR": 0.0005, "WD": 1e-4, "DR": 0.4},
	{"paramID": 103, "LR": 3e-3,  "WD": 1e-6, "DR": 0.4},
]

SINGLEVIEW_PARAM_GRID = [
	#{"paramID": 203, "LR": 0.0003, "WD": 1e-4, "DR": 0.3},
	{"paramID": 201, "LR": 0.0005, "WD": 1e-5, "DR": 0.4},
	#{"paramID": 202, "LR": 0.0007, "WD": 1e-6, "DR": 0.2},
]

RESNET_PARAM_GRID = [
	#{"paramID": 303, "LR": 0.0005, "WD": 1e-4, "DR": 0.6},
	{"paramID": 301, "LR": 0.001,  "WD": 1e-5, "DR": 0.5},
	#{"paramID": 302, "LR": 0.005,  "WD": 1e-5, "DR": 0.4},
]

# --- Main script to generate the JSON file ---
all_experiments = []
experiment_id_counter = 1
num_outer_folds = 4
num_inner_folds = 3
epochs =  50
threshold = 0.5

# 1. Add experiments for MetadataMLP
for params in MLP_PARAM_GRID:
	for fold_id in range(num_outer_folds):
		experiment = {
			"Model": "MLP_META",
			"ExpID": experiment_id_counter,
			"OUTER_FOLD": fold_id,
			#"INNER_FOLD": 0,
			"hypers": {
				"HPset": params["paramID"],
				"LR": params["LR"],
				"WD": params["WD"],
				"DR": params["DR"],
				"TH": threshold,
				"P": 5,
				"epochs": epochs,
			},
			"trained": False,
			"evaluated": False
		}
		all_experiments.append(experiment)
		experiment_id_counter += 1

# 2. Add experiments for SingleViewClassifier (for each view)
for view in ["Axial", "Sagittal", "Coronal"]:
	for params in SINGLEVIEW_PARAM_GRID:
		for fold_id in range(num_outer_folds):
			experiment = {
				"Model": f"SingleView_{view}",
				"ExpID": experiment_id_counter,
				"OUTER_FOLD": fold_id,
				#"INNER_FOLD": 0,
				"hypers": {
					"HPset": params["paramID"],
					"LR": params["LR"],
					"WD": params["WD"],
					"DR": params["DR"],
					"TH": threshold,
					"P": 5,
					"epochs": epochs,
				},
				"trained": False,
				"evaluated": False
			}
			all_experiments.append(experiment)
			experiment_id_counter += 1

# 3. Add experiments for ResNet-18

for params in RESNET_PARAM_GRID:
	for fold_id in range(num_outer_folds):
		experiment = {
			"Model": f"ResNet18",
			"ExpID": experiment_id_counter,
			"OUTER_FOLD": fold_id,
			#"INNER_FOLD": 0,
			"hypers": {
				"HPset": params["paramID"],
				"LR": params["LR"],
				"WD": params["WD"],
				"DR": params["DR"],
				"TH": threshold,
				"P": 5,
				"epochs": epochs,
			},
			"trained": False,
			"evaluated": False
		}
		all_experiments.append(experiment)
		experiment_id_counter += 1

# Save the complete list of experiments to a JSON file
with open("NCV_4_3_folds/benchmark_experiments.json", "w") as f:
	json.dump(all_experiments, f, indent=2)


In [ ]:


for experiment in INNER_experiments:
	if experiment['trained'] == True: continue
	OUT = experiment['OUTER_FOLD']
	INN = experiment['INNER_FOLD']

	if experiment['Model'] == "RESNET_18":
		print("RES")
		#DR = hypers['DR']
		OUT = experiment['OUTER_FOLD']
		INN = experiment['INNER_FOLD']
		train_loader, val_loader = DL.create_inner_loaders(OUT, INN)
		#model = MetadataMLP(hypers['DR'])
		#best_val_loss = train_INNER_MLP(model, train_loader, val_loader, experiment)
		best_val_loss = train_INNER_RESNET(train_loader, val_loader, experiment)
		experiment['best_val_loss'] = best_val_loss
		experiment['trained'] = True
		print(f"RESNET--->   BEST_VAL  {best_val_loss}")
		save_to_json(INNER_experiments, filename="")
	elif experiment['Model'] == "MLP_META":
		print("MLP")
		hypers = experiment['hypers']
		#DR = hypers['DR']
		OUT = experiment['OUTER_FOLD']
		INN = experiment['INNER_FOLD']
		train_loader, val_loader = DL.create_inner_loaders(OUT, INN)
		model = MetadataMLP(hypers['DR'])
		best_val_loss = train_INNER_MLP(model, train_loader, val_loader, experiment)
		#best_val_loss = train_INNER_RESNET(train_loader, val_loader, experiment)
		experiment['best_val_loss'] = best_val_loss
		experiment['trained'] = True
		print(f"MLP--->   BEST_VAL   {best_val_loss}")
		save_to_json(INNER_experiments, filename="")





Loaded training/INNER_FOLDS.json.
{'ExpID': 62, 'Model': 'MLP_META', 'OUTER_FOLD': 0, 'INNER_FOLD': 0, 'hypers': {'HPset': 2, 'LR': 0.0005, 'WD': 0.0001, 'DR': 0.3, 'TH': 0.4, 'P': 5, 'Epochs': 30}, 'trained': False}
MLP
	↳ Experiment 62 | Training model... 


KeyboardInterrupt: 